# 01 – Data Understanding and Cleaning (Stages 2 & 3)
Audit the raw student-performance dataset BEFORE cleaning, apply documented
cleaning decisions, then re-audit AFTER cleaning to show quality improved.

In [ ]:
import sys
from pathlib import Path
# make the shared module in ../src importable (no personal paths)
sys.path.append(str(Path.cwd().parent / 'src'))

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from common import *

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})

## 1. Load raw data and first look

In [ ]:
raw = pd.read_csv(RAW_PATH, low_memory=False)
print("Shape:", raw.shape)
print(raw.dtypes.value_counts())
print(raw[TARGET].value_counts())

## 2. Quality audit BEFORE cleaning

In [ ]:
def audit(df):
    return {
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "total_missing_cells": int(df.isna().sum().sum()),
        "rows_with_any_missing": int(df.isna().any(axis=1).sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_student_ids": int(df["student_id"].duplicated().sum()),
        "program_labels": int(df["program"].nunique()),
        "impossible_values": int(sum(((df[c] < lo) | (df[c] > hi)).sum() for c, (lo, hi) in VALID_RANGES.items())),
    }

before = audit(raw)
print(before)

miss = (raw.isna().sum().to_frame("missing").assign(pct=lambda d: (d.missing / len(raw) * 100).round(2))
        .query("missing > 0").sort_values("missing", ascending=False))
miss.to_csv(RES_DIR / "missing_before.csv")
print(miss)

imp_rows = []
for c, (lo, hi) in VALID_RANGES.items():
    n = int(((raw[c] < lo) | (raw[c] > hi)).sum())
    if n:
        imp_rows.append({"column": c, "valid_range": f"{lo}-{hi}", "invalid_count": n,
                         "min_seen": raw[c].min(), "max_seen": raw[c].max()})
imp = pd.DataFrame(imp_rows); imp.to_csv(RES_DIR / "impossible_values.csv", index=False); print(imp)

prog_before = raw["program"].value_counts()
prog_before.to_csv(RES_DIR / "program_labels_before.csv")
print(prog_before)

### Figure 1 – missing values per column

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
m = miss.sort_values("pct")
ax.barh(m.index, m["pct"], color="#2E6FA3")
for i, v in enumerate(m["pct"]):
    ax.text(v + 0.03, i, f"{v:.1f}%", va="center", fontsize=8)
ax.set_xlabel("Missing values (%)"); ax.set_title("Missing values per column (raw data)")
plt.tight_layout(); plt.savefig(FIG_DIR / "fig01_missing_values.png"); plt.close()

## 3. Cleaning decisions (each step is logged)
| Issue | Decision | Reason |
|---|---|---|
| Whitespace / case / abbreviation in `program` (13 labels for 8 programs) | strip, lower-case, map `comp sci`, title-case | same program must have one label |
| 800 duplicate records | drop | duplicates would count students twice and leak between train/test |
| Impossible age (<16 or >60), attendance >100, midterm >100 | set to NaN | cannot be real; imputing keeps the row |
| Missing values (13 numeric + `internet_access`) | impute INSIDE the ML pipeline (median / most frequent) | avoids leakage from validation data |
| IQR outliers in plausible ranges | keep | real students can study 30+ h or commute 3 h |

In [ ]:
clean, log = clean_data(raw)
print(json.dumps(log, indent=2))

## 4. Outlier check (IQR) on the cleaned data – outliers inside valid ranges are kept

In [ ]:
rows = []
for c in clean.select_dtypes("number").columns:
    s = clean[c].dropna()
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    n = int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())
    if n:
        rows.append({"column": c, "iqr_outliers": n, "pct": round(n / len(s) * 100, 2)})
out_df = pd.DataFrame(rows).sort_values("iqr_outliers", ascending=False)
out_df.to_csv(RES_DIR / "iqr_outliers.csv", index=False); print(out_df)

## 5. Quality audit AFTER cleaning (evidence that quality improved)

In [ ]:
after = audit(clean)
comparison = pd.DataFrame({"before": before, "after": after})
comparison.to_csv(RES_DIR / "quality_before_after.csv"); print(comparison)

miss_after = clean.isna().sum().to_frame("missing").query("missing > 0")
miss_after["pct"] = (miss_after.missing / len(clean) * 100).round(2)
miss_after.to_csv(RES_DIR / "missing_after.csv")

tgt = clean[TARGET].value_counts().reindex(CLASS_ORDER)
pd.DataFrame({"count": tgt, "pct": (tgt / tgt.sum() * 100).round(2)}).to_csv(RES_DIR / "target_distribution.csv")
print(tgt)

In [ ]:
clean.to_csv(CLEAN_PATH, index=False)
clean.sample(5000, random_state=RANDOM_STATE).to_csv(ROOT / "data" / "sample" / "student_performance_sample_5000.csv", index=False)
json.dump({"before": before, "after": after, "log": log}, open(RES_DIR / "cleaning_log.json", "w"), indent=2, default=int)
print("Saved clean data:", CLEAN_PATH)